In [1]:
# LABELING 01 — ENVIRONMENT & SOURCE VALIDATION
# OFFLINE ONLY | NO AUTO LABELING
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

ROOT = Path(r"C:\Penelitian Sentimen Bencana Nasional")

LABELING_DIR = ROOT / "03_Labeling"
INPUT_DIR = LABELING_DIR / "inputs"
CODEBOOK_DIR = LABELING_DIR / "codebook"
OUTPUT_DIR = LABELING_DIR / "outputs"

CODEBOOK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_PATH = INPUT_DIR / "labeling_all_381.csv"
MODEL_PATH = INPUT_DIR / "labeling_candidates_348.csv"
MANIFEST_PATH = INPUT_DIR / "cleaning_handoff_manifest.json"

for path in [ALL_PATH, MODEL_PATH, MANIFEST_PATH]:
    assert path.is_file(), f"Missing input: {path}"

df_all = pd.read_csv(
    ALL_PATH,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

df_model = pd.read_csv(
    MODEL_PATH,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

manifest = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)

assert len(df_all) == 381
assert len(df_model) == 348

assert df_all["id"].is_unique
assert df_model["id"].is_unique

assert set(df_model["id"]).issubset(set(df_all["id"]))

assert df_all["sentiment_label"].eq("").all()
assert df_model["sentiment_label"].eq("").all()

assert df_all["text_original"].str.strip().ne("").all()
assert df_all["text_semantic_verified"].str.strip().ne("").all()

assert manifest["status"] == "READY_FOR_LABELING"
assert manifest["total_eligible_tweets"] == 381
assert manifest["model_candidates"] == 348

EXPECTED_CASES = {
    "lombok_2018": 114,
    "palu_2018": 118,
    "sumatra_2025": 149
}

actual_cases = df_all["case"].value_counts().to_dict()
assert actual_cases == EXPECTED_CASES, actual_cases

# Check semua teks kandidat masih identik dengan
# sumber ALL_381 berdasarkan ID.
source_text = df_all.set_index("id")["text_linguistic"]
candidate_text = df_model.set_index("id")["text_linguistic"]

assert candidate_text.equals(
    source_text.loc[candidate_text.index]
)

print("=" * 60)
print("LABELING 01 — SOURCE VALIDATION")
print("=" * 60)
print("Total eligible tweets :", len(df_all))
print("Model candidates      :", len(df_model))
print("Unassigned labels     :", df_all["sentiment_label"].eq("").sum())
print("Cases                 :", actual_cases)
print("Source integrity      : PASS")
print("No data modification.")


LABELING 01 — SOURCE VALIDATION
Total eligible tweets : 381
Model candidates      : 348
Unassigned labels     : 381
Cases                 : {'sumatra_2025': 149, 'palu_2018': 118, 'lombok_2018': 114}
Source integrity      : PASS
No data modification.


In [2]:
# LABELING 02 — SENTIMENT CODEBOOK v1.0
# CREATE DOCUMENTATION | NO SENTIMENT INFERENCE
# ============================================================

CODEBOOK_PATH = (
    CODEBOOK_DIR / "sentiment_codebook_v1.md"
)

assert not CODEBOOK_PATH.exists(), (
    f"Codebook sudah ada: {CODEBOOK_PATH}"
)

codebook = """# Sentiment Labeling Codebook v1.0

## 1. Tujuan
Mengidentifikasi sikap penulis terhadap penetapan status
bencana nasional pada Lombok 2018, Palu 2018,
dan Sumatera 2025.

## 2. Unit Analisis
Satu postingan/tweet dengan tweet ID unik.

Setiap tweet diberi tepat satu label akhir.

## 3. Objek Sentimen
Kebijakan atau keputusan penetapan status bencana nasional.

Penilaian bukan didasarkan pada sentimen umum terhadap
bencana, korban, maupun pemerintah.

## 4. Kategori

### POSITIF
Dukungan, persetujuan, desakan, tuntutan, atau kritik
yang secara jelas mendukung penetapan status
bencana nasional.

### NEGATIF
Penolakan, ketidaksetujuan, atau pernyataan bahwa
penetapan status bencana nasional tidak diperlukan.

### NETRAL
Berita, laporan, pertanyaan informatif, kutipan tanpa
persetujuan penulis, atau unggahan tanpa sikap jelas
terhadap penetapan status bencana nasional.

## 5. Prinsip Anotasi
1. Utamakan maksud penulis, bukan hanya keyword.
2. Jangan samakan kesedihan akibat bencana dengan POSITIF.
3. Jangan samakan kritik kepada pemerintah dengan POSITIF
   kecuali terkait dukungan terhadap penetapan status.
4. Bedakan opini penulis dari kutipan atau laporan berita.
5. Pertanyaan retoris dinilai dari konteks.
6. Jika penulis mendukung sekaligus menolak dan sikap
   dominan tidak jelas, tandai untuk review.
7. Gunakan text_original dan text_semantic_verified
   untuk memahami konteks.
8. Jangan menggunakan label hasil prediksi model
   sebagai pengganti keputusan anotator.
9. Jangan mengubah teks sumber ketika memberikan label.
10. Label akhir hanya POSITIF, NEGATIF, atau NETRAL.

## 6. Status Review
PENDING: belum dianotasi.
LABELED: label telah diberikan.
AMBIGUOUS: membutuhkan adjudikasi.
ADJUDICATED: kasus telah diputuskan melalui review.

AMBIGUOUS bukan kelas sentimen keempat.

## 7. Contoh Ilustratif
- "Segera tetapkan Palu sebagai bencana nasional."
  Label: POSITIF

- "Tidak perlu menetapkan status bencana nasional."
  Label: NEGATIF

- "Pemerintah menjelaskan alasan status belum ditetapkan."
  Label: NETRAL

- "Mengapa Lombok belum menjadi bencana nasional?"
  Label: NETRAL jika pertanyaan informatif.
  Review apabila konteks mengisyaratkan desakan.

## 8. Konsistensi
- Satu ID harus memiliki satu label final.
- Konten identik perlu diperiksa konsistensinya.
- Label untuk kandidat modeling diturunkan dari ALL_381.
- Semua keputusan ambigu harus diselesaikan sebelum
  ekspor dataset final.

## 9. Versi
Version: 1.0
Status: DRAFT_FOR_PILOT
"""

CODEBOOK_PATH.write_text(
    codebook,
    encoding="utf-8"
)

required_sections = [
    "POSITIF", "NEGATIF", "NETRAL",
    "AMBIGUOUS", "text_original",
    "text_semantic_verified",
    "DRAFT_FOR_PILOT"
]

saved = CODEBOOK_PATH.read_text(encoding="utf-8")

assert all(
    section in saved for section in required_sections
)

print("=" * 60)
print("LABELING 02 — CODEBOOK CREATION")
print("=" * 60)
print("File       :", CODEBOOK_PATH)
print("Version    : 1.0")
print("Sentiments : POSITIF, NEGATIF, NETRAL")
print("Status     : DRAFT_FOR_PILOT")
print("CODEBOOK VALIDATION: PASS")


LABELING 02 — CODEBOOK CREATION
File       : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\codebook\sentiment_codebook_v1.md
Version    : 1.0
Sentiments : POSITIF, NEGATIF, NETRAL
Status     : DRAFT_FOR_PILOT
CODEBOOK VALIDATION: PASS


In [3]:
# LABELING 03 — CODEBOOK CONSISTENCY TEST
# ILLUSTRATIVE CASES ONLY | NO MODEL PREDICTION
# ============================================================

examples = [
    {
        "text": "Tetapkan banjir Sumatera sebagai bencana nasional.",
        "expected_label": "POSITIF",
        "reason": "Desakan langsung"
    },
    {
        "text": "Palu seharusnya segera berstatus bencana nasional.",
        "expected_label": "POSITIF",
        "reason": "Dukungan jelas"
    },
    {
        "text": "Tidak perlu menaikkan status menjadi bencana nasional.",
        "expected_label": "NEGATIF",
        "reason": "Penolakan eksplisit"
    },
    {
        "text": "Status nasional bukan langkah yang diperlukan.",
        "expected_label": "NEGATIF",
        "reason": "Menilai penetapan tidak perlu"
    },
    {
        "text": "BNPB membahas status bencana nasional Lombok.",
        "expected_label": "NETRAL",
        "reason": "Informasi tanpa sikap penulis"
    },
    {
        "text": "Presiden menjelaskan alasan status belum ditetapkan.",
        "expected_label": "NETRAL",
        "reason": "Laporan pernyataan"
    },
    {
        "text": "Mengapa status bencana nasional belum ditetapkan?",
        "expected_label": "REVIEW",
        "reason": "Pertanyaan berpotensi ambigu"
    },
]

example_df = pd.DataFrame(examples)

VALID_LABELS = {"POSITIF", "NEGATIF", "NETRAL"}
VALID_TEST_DECISIONS = VALID_LABELS | {"REVIEW"}

assert example_df["expected_label"].isin(
    VALID_TEST_DECISIONS
).all()

assert not example_df["text"].duplicated().any()

print("CODEBOOK EXAMPLE CHECK: PASS")
print("Example count:", len(example_df))
print("Final classes:", sorted(VALID_LABELS))
print("Review is a workflow status, not a fourth class.")

display(example_df)


CODEBOOK EXAMPLE CHECK: PASS
Example count: 7
Final classes: ['NEGATIF', 'NETRAL', 'POSITIF']
Review is a workflow status, not a fourth class.


,text,expected_label,reason
0,Tetapkan banjir Sumatera sebagai bencana nasio...,POSITIF,Desakan langsung
1,Palu seharusnya segera berstatus bencana nasio...,POSITIF,Dukungan jelas
2,Tidak perlu menaikkan status menjadi bencana n...,NEGATIF,Penolakan eksplisit
3,Status nasional bukan langkah yang diperlukan.,NEGATIF,Menilai penetapan tidak perlu
4,BNPB membahas status bencana nasional Lombok.,NETRAL,Informasi tanpa sikap penulis
5,Presiden menjelaskan alasan status belum ditet...,NETRAL,Laporan pernyataan
6,Mengapa status bencana nasional belum ditetapkan?,REVIEW,Pertanyaan berpotensi ambigu


In [4]:
# LABELING 04 — STRATIFIED PILOT ANNOTATION PREPARATION
# 30 TWEETS | 10 PER CASE | OFFLINE ONLY
# ============================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

assert len(df_all) == 381
assert df_all["id"].is_unique

PILOT_SEED = 42
PILOT_PER_CASE = 10

PILOT_DIR = OUTPUT_DIR / "pilot"
PILOT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_CSV = PILOT_DIR / "pilot_annotation_30.csv"
PILOT_XLSX = PILOT_DIR / "pilot_annotation_30.xlsx"
PILOT_MANIFEST = PILOT_DIR / "pilot_manifest.json"

for path in [PILOT_CSV, PILOT_XLSX, PILOT_MANIFEST]:
    if path.exists():
        raise FileExistsError(
            f"Output sudah tersedia: {path}"
        )

# ------------------------------------------------------------
# A. Pilih satu teks per grup terlebih dahulu
# agar pilot berfokus pada variasi konten.
# ------------------------------------------------------------

pool = df_all.loc[
    df_all["eligible_for_model_candidate"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
].copy()

assert len(pool) == 348

# Untuk memastikan ada variasi konten dan
# sebagian tweet sulit juga terwakili,
# pilih sampel secara deterministik.
pilot_parts = []

for case in sorted(pool["case"].unique()):
    subset = pool.loc[pool["case"] == case]

    assert len(subset) >= PILOT_PER_CASE

    selected = subset.sample(
        n=PILOT_PER_CASE,
        random_state=PILOT_SEED
    )

    pilot_parts.append(selected)

pilot = pd.concat(
    pilot_parts,
    ignore_index=True
)

assert len(pilot) == 30
assert pilot["id"].is_unique

assert pilot["case"].value_counts().to_dict() == {
    "lombok_2018": 10,
    "palu_2018": 10,
    "sumatra_2025": 10
}

# ------------------------------------------------------------
# B. Buat kolom anotasi manual
# ------------------------------------------------------------

pilot = pilot[
    [
        "case",
        "id",
        "text_original",
        "text_semantic_verified",
        "text_linguistic",
        "model_selection_status",
        "final_text_hash"
    ]
].copy()

pilot["sentiment_label"] = ""
pilot["annotation_status"] = "PENDING"
pilot["label_notes"] = ""
pilot["annotator"] = ""
pilot["labeled_at"] = ""

# ------------------------------------------------------------
# C. Simpan CSV
# ------------------------------------------------------------

pilot.to_csv(
    PILOT_CSV,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# D. Simpan workbook Excel
# ------------------------------------------------------------

from openpyxl.styles import Font, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

with pd.ExcelWriter(
    PILOT_XLSX,
    engine="openpyxl"
) as writer:

    pilot.to_excel(
        writer,
        sheet_name="PILOT_30",
        index=False
    )

    ws = writer.sheets["PILOT_30"]
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = ws.dimensions

    headers = {
        cell.value: cell.column_letter
        for cell in ws[1]
    }

    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(wrap_text=True)

    for column in ws.columns:
        name = str(column[0].value)
        letter = column[0].column_letter

        ws.column_dimensions[letter].width = (
            75 if name.startswith("text_") else 25
        )

        for cell in column[1:]:
            if isinstance(cell.value, str):
                cell.data_type = "s"

            if name == "id":
                cell.number_format = "@"

    # Dropdown sentimen:
    sentiment_col = headers["sentiment_label"]
    sentiment_validation = DataValidation(
        type="list",
        formula1='"POSITIF,NEGATIF,NETRAL"',
        allow_blank=True
    )
    ws.add_data_validation(sentiment_validation)
    sentiment_validation.add(
        f"{sentiment_col}2:{sentiment_col}31"
    )

    # Dropdown status:
    status_col = headers["annotation_status"]
    status_validation = DataValidation(
        type="list",
        formula1='"PENDING,LABELED,AMBIGUOUS,ADJUDICATED"',
        allow_blank=False
    )
    ws.add_data_validation(status_validation)
    status_validation.add(
        f"{status_col}2:{status_col}31"
    )

# ------------------------------------------------------------
# E. Manifest untuk reproduktibilitas
# ------------------------------------------------------------

pilot_manifest = {
    "stage": "03_Labeling",
    "part": "Pilot Annotation Preparation",
    "codebook_version": "1.0",
    "codebook_status": "DRAFT_FOR_PILOT",
    "seed": PILOT_SEED,
    "sampling_method": "stratified_random_by_case",
    "sample_size": len(pilot),
    "sample_per_case": PILOT_PER_CASE,
    "labels_assigned": 0,
    "status": "READY_FOR_PILOT_ANNOTATION"
}

PILOT_MANIFEST.write_text(
    json.dumps(
        pilot_manifest,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# F. Audit
# ------------------------------------------------------------

print("=" * 64)
print("LABELING 04 — PILOT ANNOTATION PREPARATION")
print("=" * 64)

print("Source tweets       :", len(df_all))
print("Model candidate pool:", len(pool))
print("Pilot sample        :", len(pilot))
print("Pilot labels filled :", 0)

print("\nDistribution:")
display(pilot["case"].value_counts())

print("\nOutput files:")
print(PILOT_CSV)
print(PILOT_XLSX)
print(PILOT_MANIFEST)

print("\nLABELING 04 PILOT PREPARATION: PASS")


LABELING 04 — PILOT ANNOTATION PREPARATION
Source tweets       : 381
Model candidate pool: 348
Pilot sample        : 30
Pilot labels filled : 0

Distribution:


case
lombok_2018     10
palu_2018       10
sumatra_2025    10
Name: count, dtype: int64


Output files:
C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\pilot\pilot_annotation_30.csv
C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\pilot\pilot_annotation_30.xlsx
C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\pilot\pilot_manifest.json

LABELING 04 PILOT PREPARATION: PASS


In [5]:
# LABELING 05 — PILOT ANNOTATION VALIDATION
# 30 TWEETS | SOURCE INTEGRITY | ADJUDICATION FLAGS
# OFFLINE ONLY | NO AUTOMATIC RELABELING
# ============================================================

import pandas as pd
from pathlib import Path
import json

PILOT_DIR = OUTPUT_DIR / "pilot"
PILOT_XLSX = PILOT_DIR / "pilot_annotation_30.xlsx"
PILOT_CSV = PILOT_DIR / "pilot_annotation_30.csv"

AUDIT_CSV = PILOT_DIR / "pilot_validation_audit.csv"
AUDIT_JSON = PILOT_DIR / "pilot_validation_summary.json"

for p in [AUDIT_CSV, AUDIT_JSON]:
    if p.exists():
        raise FileExistsError(f"Output already exists: {p}")

# A. Read source snapshot and manually edited workbook
source = pd.read_csv(
    PILOT_CSV,
    dtype={"id": "string"},
    keep_default_na=False,
    encoding="utf-8-sig"
)

reviewed = pd.read_excel(
    PILOT_XLSX,
    sheet_name="PILOT_30",
    dtype={"id": "string"},
    keep_default_na=False,
    engine="openpyxl"
)

assert len(source) == len(reviewed) == 30
assert source["id"].is_unique
assert reviewed["id"].is_unique

# B. Verify identities and immutable columns
assert set(source["id"]) == set(reviewed["id"])

immutable_cols = [
    "case",
    "text_original",
    "text_semantic_verified",
    "text_linguistic",
    "model_selection_status",
    "final_text_hash"
]

source_indexed = source.set_index("id")
reviewed_indexed = reviewed.set_index("id")

for col in immutable_cols:
    expected = source_indexed[col].astype("string")
    actual = reviewed_indexed[col].astype("string")

    assert actual.sort_index().equals(
        expected.sort_index()
    ), f"Source integrity mismatch: {col}"

# C. Normalize annotation fields for validation
reviewed["sentiment_label"] = (
    reviewed["sentiment_label"]
    .astype("string").str.strip().str.upper()
)

reviewed["annotation_status"] = (
    reviewed["annotation_status"]
    .astype("string").str.strip().str.upper()
)

allowed_labels = {"POSITIF", "NEGATIF", "NETRAL"}

assert reviewed["sentiment_label"].isin(
    allowed_labels
).all(), "Invalid or missing sentiment label."

assert reviewed["annotation_status"].eq(
    "LABELED"
).all(), "Some rows are not LABELED."

for col in ["label_notes", "annotator", "labeled_at"]:
    assert reviewed[col].astype("string").str.strip().ne(
        ""
    ).all(), f"Missing values in: {col}"

dates = pd.to_datetime(
    reviewed["labeled_at"],
    errors="coerce"
)

assert dates.notna().all(), "Invalid annotation dates."

# D. Candidate semantic review:
# Manual audit targets, NOT automatic corrections.
REVIEW_IDS = {
    "1028865436794810368": (
        "Penolakan atau kritik terhadap pemerintah?"
    ),
    "1046304416356130816": (
        "Pertanyaan atau dukungan eksplisit?"
    ),
    "1047236117546774528": (
        "Judul berita atau dukungan penulis?"
    ),
    "1996474933816529181": (
        "Laporan desakan pihak lain atau sikap penulis?"
    )
}

reviewed["needs_adjudication"] = (
    reviewed["id"].isin(REVIEW_IDS)
)

reviewed["adjudication_reason"] = (
    reviewed["id"].map(REVIEW_IDS).fillna("")
)

assert reviewed["needs_adjudication"].sum() == 4

# E. Distribution
distribution = (
    reviewed.groupby(
        ["case", "sentiment_label"]
    )
    .size()
    .unstack(fill_value=0)
)

print("LABEL DISTRIBUTION BY CASE")
display(distribution)

print("\nSEMANTIC ADJUDICATION CANDIDATES")
display(
    reviewed.loc[
        reviewed["needs_adjudication"],
        [
            "case",
            "id",
            "text_original",
            "text_semantic_verified",
            "sentiment_label",
            "label_notes",
            "adjudication_reason"
        ]
    ]
)

# F. Save read-only derived audit report
reviewed.to_csv(
    AUDIT_CSV,
    index=False,
    encoding="utf-8-sig"
)

summary = {
    "total_pilot_tweets": len(reviewed),
    "labeled": int(
        reviewed["annotation_status"].eq("LABELED").sum()
    ),
    "positive": int(
        reviewed["sentiment_label"].eq("POSITIF").sum()
    ),
    "negative": int(
        reviewed["sentiment_label"].eq("NEGATIF").sum()
    ),
    "neutral": int(
        reviewed["sentiment_label"].eq("NETRAL").sum()
    ),
    "source_integrity": "PASS",
    "manual_adjudication_candidates": int(
        reviewed["needs_adjudication"].sum()
    ),
    "status": "ADJUDICATION_PENDING"
}

AUDIT_JSON.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\n" + "=" * 62)
print("LABELING 05 — PILOT VALIDATION")
print("=" * 62)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nValidation completed.")
print("Original files unchanged.")
print("No automatic label modification.")


LABEL DISTRIBUTION BY CASE


sentiment_label,NEGATIF,NETRAL,POSITIF
case,,,
lombok_2018,2,5,3
palu_2018,1,7,2
sumatra_2025,0,4,6



SEMANTIC ADJUDICATION CANDIDATES


,case,id,text_original,text_semantic_verified,sentiment_label,label_notes,adjudication_reason
6,lombok_2018,1028865436794810368,"Sampai saat ini, bantuan gempa Lombok, masih t...","Sampai saat ini, bantuan gempa Lombok, masih t...",NEGATIF,menyatakan bahwa pemerintah masih belum lumpuh...,Penolakan atau kritik terhadap pemerintah?
15,palu_2018,1046304416356130816,Apakah bapak Presiden juga\nsiap membantu kami...,Apakah bapak Presiden juga siap membantu kami ...,POSITIF,menunjukkan pertanyaan terhadap kesiapan pemer...,Pertanyaan atau dukungan eksplisit?
17,palu_2018,1047236117546774528,PKS Desak Pemerintah Tetapkan Gempa Palu Benca...,PKS Desak Pemerintah Tetapkan Gempa Palu Benca...,POSITIF,menyatakan desakan kepada pemerintah,Judul berita atau dukungan penulis?
24,sumatra_2025,1996474933816529181,Melihat banjir bandang yang memporak-porandaka...,Melihat banjir bandang yang memporak-porandaka...,POSITIF,berisi informasi yang menyatakan desakan untuk...,Laporan desakan pihak lain atau sikap penulis?



LABELING 05 — PILOT VALIDATION
total_pilot_tweets: 30
labeled: 30
positive: 11
negative: 3
neutral: 16
source_integrity: PASS
manual_adjudication_candidates: 4
status: ADJUDICATION_PENDING

Validation completed.
Original files unchanged.
No automatic label modification.


In [6]:
# LABELING 06 — PILOT ADJUDICATION PREPARATION
# 4 FLAGGED TWEETS | NO AUTOMATIC RELABELING
# ============================================================

from pathlib import Path
import pandas as pd
from openpyxl.styles import Font, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

PILOT_DIR = OUTPUT_DIR / "pilot"
AUDIT_PATH = PILOT_DIR / "pilot_validation_audit.csv"
ADJUDICATION_PATH = (
    PILOT_DIR / "pilot_adjudication_4.xlsx"
)

assert AUDIT_PATH.exists(), "Audit Part4 tidak ditemukan."
assert not ADJUDICATION_PATH.exists(), (
    "Workbook adjudication sudah ada. Jangan overwrite."
)

audit = pd.read_csv(
    AUDIT_PATH,
    dtype={"id": "string"},
    encoding="utf-8-sig",
    keep_default_na=False
)

flagged = audit.loc[
    audit["needs_adjudication"]
    .astype(str).str.lower().eq("true")
].copy()

assert len(flagged) == 4
assert flagged["id"].is_unique

columns = [
    "case",
    "id",
    "text_original",
    "text_semantic_verified",
    "sentiment_label",
    "label_notes",
    "adjudication_reason"
]

review = flagged[columns].copy()

review = review.rename(columns={
    "sentiment_label": "previous_label",
    "label_notes": "previous_label_notes"
})

review["final_label"] = ""
review["adjudication_decision"] = ""
review["adjudication_notes"] = ""
review["reviewer"] = ""
review["reviewed_at"] = ""

with pd.ExcelWriter(
    ADJUDICATION_PATH,
    engine="openpyxl"
) as writer:

    review.to_excel(
        writer,
        sheet_name="ADJUDICATION_4",
        index=False
    )

    ws = writer.sheets["ADJUDICATION_4"]
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = ws.dimensions

    headers = {
        cell.value: cell.column_letter
        for cell in ws[1]
    }

    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(wrap_text=True)

    for col in ws.columns:
        name = str(col[0].value)
        letter = col[0].column_letter

        ws.column_dimensions[letter].width = (
            80 if name.startswith("text_") else 30
        )

        for cell in col[1:]:
            if isinstance(cell.value, str):
                cell.data_type = "s"
            if name == "id":
                cell.number_format = "@"

    label_validation = DataValidation(
        type="list",
        formula1='"POSITIF,NEGATIF,NETRAL"',
        allow_blank=True
    )
    ws.add_data_validation(label_validation)
    label_validation.add(
        f"{headers['final_label']}2:"
        f"{headers['final_label']}5"
    )

    decision_validation = DataValidation(
        type="list",
        formula1='"KEEP,CHANGE,UNRESOLVED"',
        allow_blank=True
    )
    ws.add_data_validation(decision_validation)
    decision_validation.add(
        f"{headers['adjudication_decision']}2:"
        f"{headers['adjudication_decision']}5"
    )

print("=" * 60)
print("LABELING 06 — ADJUDICATION PREPARATION")
print("=" * 60)
print("Flagged tweets :", len(review))
print("Source records :", len(audit))
print("Workbook       :", ADJUDICATION_PATH)
print("Auto relabeling: OFF")
print("ADJUDICATION PREPARATION: PASS")

display(
    review[
        ["case", "id", "previous_label",
         "adjudication_reason"]
    ]
)


LABELING 06 — ADJUDICATION PREPARATION
Flagged tweets : 4
Source records : 30
Workbook       : C:\Penelitian Sentimen Bencana Nasional\03_Labeling\outputs\pilot\pilot_adjudication_4.xlsx
Auto relabeling: OFF
ADJUDICATION PREPARATION: PASS


,case,id,previous_label,adjudication_reason
6,lombok_2018,1028865436794810368,NEGATIF,Penolakan atau kritik terhadap pemerintah?
15,palu_2018,1046304416356130816,POSITIF,Pertanyaan atau dukungan eksplisit?
17,palu_2018,1047236117546774528,POSITIF,Judul berita atau dukungan penulis?
24,sumatra_2025,1996474933816529181,POSITIF,Laporan desakan pihak lain atau sikap penulis?
